# job.am IT Jobs Scraper

**Source:** [job.am](https://job.am) — Armenian job board  
**Category:** Information Technologies, Software Development (category I=17)  
**Purpose:** Scrape all IT job postings with full text for NLP skill extraction.

### Scraping strategy
1. Collect job URLs from the IT category listing (`/en/search/jobs?I=17`)
2. Supplement with keyword searches (`developer`, `programmer`, `software`, `engineer`)
3. Filter out non-IT jobs by title keywords and industry field
4. Scrape each detail page: extract all structured sections

### Page structure
- Listing page: standard HTML with `jobs-card` divs
- Detail page: `section.companyedit-page` → h3-separated sections (English jobs) or `<strong>`-based sections (Armenian jobs)
- Fields: Title, Company, Location, Employment type, Experience level, Industry, Deadline, Description, Responsibilities, Requirements, Additional Notes

### Ethics & robots.txt
- `robots.txt` says `Allow: /` for all bots (checked 2026-03-22)  
- Only disallows `/API/*`, `/search/topjobs`, `/ads/` — we avoid those  
- Rate-limited: 1.5 s between requests  

### Output files
- `data/raw/jobs/jobam_jobs_raw.csv` — all extracted fields
- `data/processed/jobs/jobam_jobs_standardized.csv` — canonical NLP schema

In [ ]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import date
from pathlib import Path

BASE_URL = "https://job.am"
HEADERS  = {
    "User-Agent": "Mozilla/5.0 (compatible; ThesisResearch/1.0; Armenian IT curriculum alignment; academic use)",
    "Accept-Language": "en-US,en;q=0.9",
}
DELAY_S  = 1.5
TODAY    = date.today().isoformat()

BASE_DIR = Path.cwd()
RAW_DIR  = BASE_DIR / "data" / "raw" / "jobs"
PROC_DIR = BASE_DIR / "data" / "processed" / "jobs"

# IT-relevance keywords for title-based filter
IT_TITLE_KEYWORDS = [
    "developer", "engineer", "programmer", "software", "backend", "frontend",
    "front-end", "back-end", "full stack", "fullstack", "devops", "data science",
    "data analyst", "machine learning", "ai ", "python", "java ", "javascript",
    "react", "node", ".net", "php", "qa ", "quality assurance", "tester",
    "system admin", "sysadmin", "network", "security", "cybersecurity", "cloud",
    "database", "sql", "it support", "it specialist", "1c", "web ", "mobile",
    "android", "ios", "flutter", "bitrix", "scrum", "agile",
    "cragravorogh", "cragravoric", "texnakan",
]
IT_INDUSTRY_KEYWORD = "information technologies"

print(f"Today: {TODAY}")
print(f"Output: {PROC_DIR}")

## Helper functions

In [ ]:
def is_it_relevant(title, industry):
    title_lower = title.lower()
    if IT_INDUSTRY_KEYWORD in industry.lower():
        return True
    return any(kw in title_lower for kw in IT_TITLE_KEYWORDS)

def get_listing_links(url):
    """Fetch one listing/search page and return unique /en/job/ links."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
    except Exception as e:
        print(f"  ERROR {url}: {e}")
        return []
    soup = BeautifulSoup(resp.text, "html.parser")
    links = list(dict.fromkeys(
        a["href"] for a in soup.find_all("a", href=True) if "/en/job/" in a["href"]
    ))
    return [BASE_URL + l if l.startswith("/") else l for l in links]

# Armenian section header equivalents (built via chr() to avoid encoding issues)
AM_HEADINGS = {
    "description": "".join(chr(c) for c in [
        0x546,0x56f,0x561,0x580,0x561,0x563,0x580,0x578,0x582,0x569,0x575,0x578,0x582,0x576
    ]),
    "responsibilities": "".join(chr(c) for c in [
        0x54a,0x561,0x580,0x57f,0x561,0x56f,0x561,0x576,0x578,0x582,0x569,0x575,0x578,0x582,0x576,0x576,0x565,0x580
    ]),
    "requirements": "".join(chr(c) for c in [
        0x54a,0x561,0x570,0x561,0x576,0x57b,0x576,0x565,0x580
    ]),
    "additional notes": "".join(chr(c) for c in [
        0x540,0x561,0x57e,0x565,0x56c,0x575,0x561,0x56c,0x20,
        0x576,0x577,0x578,0x582,0x574,0x576,0x565,0x580
    ]),
}

def text_after_h3(section, heading_text):
    """Get text after h3 matching English or Armenian heading, up to next h3."""
    am_equiv = AM_HEADINGS.get(heading_text.lower(), "")
    h3 = section.find("h3", string=lambda s: s and (
        heading_text.lower() in s.lower() or (am_equiv and am_equiv in s)
    ))
    if not h3:
        return ""
    texts = []
    for sib in h3.find_next_siblings():
        if sib.name == "h3":
            break
        t = sib.get_text(" ", strip=True)
        if t:
            texts.append(t)
    return " ".join(texts)

def extract_meta_field(raw_text, label):
    """Extract a labeled metadata field from raw section text."""
    stop = r"(?:Location|Salary|Employment type|Work schedule|Work experience|Industry|Deadline|Description|Log in|$)"
    pattern = rf"{re.escape(label)}[:\s]+(.+?)(?={stop})"
    m = re.search(pattern, raw_text)
    return m.group(1).strip() if m else ""

print("Helpers defined.")

## Step 1 — Collect job URLs

In [ ]:
listing_sources = [
    ("IT category (I=17)",  f"{BASE_URL}/en/search/jobs?I=17"),
    ("keyword=developer",   f"{BASE_URL}/en/jobs?q=developer"),
    ("keyword=programmer",  f"{BASE_URL}/en/jobs?q=programmer"),
    ("keyword=software",    f"{BASE_URL}/en/jobs?q=software"),
    ("keyword=engineer",    f"{BASE_URL}/en/jobs?q=engineer"),
]

all_links = []
seen = set()
for name, url in listing_sources:
    links = get_listing_links(url)
    new = [l for l in links if l not in seen]
    seen.update(new)
    all_links.extend(new)
    print(f"  {name}: {len(links)} found, {len(new)} new → total {len(all_links)}")
    time.sleep(DELAY_S)

print(f"\nTotal unique job URLs: {len(all_links)}")

## Step 2 — Scrape detail pages

Extracts from each job page:
- **Metadata**: company, location, salary, employment type, experience level, industry, deadline
- **Description** — intro paragraph
- **Responsibilities** — what you will do
- **Requirements** — qualifications and skills needed (primary NLP target)
- **Additional Notes** — preferred skills, benefits

Handles both English jobs (h3-separated) and Armenian jobs (strong-separated).  
Non-IT jobs (not in IT industry AND no IT keywords in title) are filtered out.

In [ ]:
records = []
skipped_not_it = 0
errors = 0

for i, url in enumerate(all_links, 1):
    print(f"[{i}/{len(all_links)}] {url}")
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
    except Exception as e:
        print(f"  ERROR: {e}")
        errors += 1
        time.sleep(DELAY_S)
        continue

    soup = BeautifulSoup(resp.text, "html.parser")
    section = soup.find("section", class_="companyedit-page")
    if not section:
        print(f"  No companyedit-page section")
        errors += 1
        time.sleep(DELAY_S)
        continue

    h1 = soup.find("h1")
    job_title = h1.get_text(strip=True) if h1 else ""

    company_a = section.find("a", href=lambda h: h and "/en/company/" in str(h))
    company_name = company_a.get_text(strip=True) if company_a else ""

    raw = section.get_text(" ", strip=True)
    location        = extract_meta_field(raw, "Location")
    salary          = extract_meta_field(raw, "Salary")
    employment_type = extract_meta_field(raw, "Employment type")
    work_schedule   = extract_meta_field(raw, "Work schedule")
    experience_level = extract_meta_field(raw, "Work experience")
    industry        = extract_meta_field(raw, "Industry")
    deadline_raw    = extract_meta_field(raw, "Deadline")

    # Normalize deadline M/D/YYYY → YYYY-MM-DD
    deadline = ""
    if deadline_raw:
        dm = re.match(r"(\d+)/(\d+)/(\d{4})", deadline_raw)
        deadline = f"{dm.group(3)}-{int(dm.group(1)):02d}-{int(dm.group(2)):02d}" if dm else deadline_raw

    # IT relevance filter
    if not is_it_relevant(job_title, industry):
        print(f"  SKIP (not IT): {job_title}")
        skipped_not_it += 1
        time.sleep(DELAY_S)
        continue

    description      = text_after_h3(section, "Description")
    responsibilities = text_after_h3(section, "Responsibilities")
    requirements     = text_after_h3(section, "Requirements")
    additional_notes = text_after_h3(section, "Additional Notes")

    parts = [p for p in [description, responsibilities, requirements, additional_notes] if p]
    full_text = "\n\n".join(parts)

    # Fallback: unstructured jobs use <strong> headers, grab work-description div
    if not full_text:
        work_desc = section.find("div", class_=lambda c: c and "work-description" in str(c))
        if work_desc:
            full_text = work_desc.get_text(" ", strip=True)

    records.append({
        "source":            "job.am",
        "source_url":        url,
        "job_title":         job_title,
        "company_name":      company_name,
        "location":          location,
        "employment_type":   employment_type,
        "work_schedule":     work_schedule,
        "experience_level":  experience_level,
        "industry":          industry,
        "salary":            salary,
        "deadline":          deadline,
        "description":       description,
        "responsibilities":  responsibilities,
        "requirements":      requirements,
        "additional_notes":  additional_notes,
        "full_text":         full_text,
    })
    time.sleep(DELAY_S)

print(f"\nIT-relevant: {len(records)} | Skipped (not IT): {skipped_not_it} | Errors: {errors}")

## Step 3 — Save and review

In [ ]:
df = pd.DataFrame(records)

# Raw
raw_path = RAW_DIR / "jobam_jobs_raw.csv"
df.to_csv(raw_path, index=False, encoding="utf-8")
print(f"Raw: {raw_path}  ({len(df)} rows)")

# Standardized
std_cols = [
    "source", "source_url", "job_title", "company_name", "location",
    "employment_type", "experience_level", "salary", "deadline",
    "description", "responsibilities", "requirements", "additional_notes",
    "full_text",
]
std_df = df[[c for c in std_cols if c in df.columns]]
std_path = PROC_DIR / "jobam_jobs_standardized.csv"
std_df.to_csv(std_path, index=False, encoding="utf-8")
print(f"Standardized: {std_path}  ({len(std_df)} rows)")

In [ ]:
print("=== FIELD COVERAGE ===")
for col in std_df.columns:
    filled = std_df[col].replace("", pd.NA).notna().sum()
    pct = filled / len(std_df) * 100
    print(f"  {col:30s}: {filled:3d}/{len(std_df)} ({pct:.0f}%)")

print()
ft = std_df["full_text"].replace("", pd.NA).dropna().str.len()
print(f"full_text — Min: {ft.min():.0f}  Median: {ft.median():.0f}  Max: {ft.max():.0f}")
print()
std_df[["job_title","company_name","experience_level","full_text"]].assign(
    full_text_len=std_df["full_text"].str.len()
).drop(columns=["full_text"])